Author: Jasmine Sun

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

# Reading data

In [2]:
data_path = "/Users/jasminesun/Desktop/Data/CAR_-_EP_Flow_Activity_Queue__Agent_Names"
adhoc_data_path = "/Users/jasminesun/Desktop/Data/"

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])


In [4]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
print("Data files read = ",i)

Data files read =  53


In [5]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [6]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]

In [7]:
df_main.shape

(3328626, 8)

In [8]:
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [9]:
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour


In [10]:
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)

In [11]:
df_main.reset_index(inplace = True, drop = True)

In [12]:
df_main['Date'] = df_main['Activity Start Timestamp'].dt.date

In [13]:
df_main['Date'] = pd.to_datetime(df_main['Date'], errors='coerce')

# 2️⃣ Then safely extract the numeric weekday
df_main['DayOfWeek'] = df_main['Date'].dt.dayofweek + 1

In [14]:
df_main['MonthNum'] = df_main['Date'].dt.month

# Data prep for dashboard
## Data for the Main Menu Traffic-- Seasonal, Weekly and Hourly trends tabs of the dashboard 
#### Link (use LegalAid PowerBI Account to access): https://app.powerbi.com/links/sxqLSPcjts?ctid=cc161ac2-10c5-420b-9b80-66b398123070&pbi_source=linkShare&bookmarkGuid=196e4a2f-e3cf-4e7e-8029-cec9a53fa216

In [46]:
def dash_data(ep_name):
    mainmenu_rows = df_main.loc[df_main['EP Name'] == ep_name, :]
    mainmenu_data = df_main.loc[df_main['Contact Session ID'].isin(mainmenu_rows['Contact Session ID']), :]
    mainmenu_activity = mainmenu_data[['Contact Session ID', 'Date', 'hour', 'Activity Name', 'DayOfWeek']].dropna(subset=["Activity Name"]).copy()
    mainmenu_activity.dropna(subset=["Activity Name"], inplace=True)

    to_drop = ['SeniorsConfirmationMenu', 'SuburbsOrCityMenu', 'LegalMenu2']
    mainmenu_activity = mainmenu_activity[~mainmenu_activity['Activity Name'].isin(to_drop)]
    
    # Ensure order within each session
    mainmenu_activity = mainmenu_activity.sort_values(["Contact Session ID", "Date", "hour"])

    # Group by session
    g = mainmenu_activity.groupby("Contact Session ID")

    # Assign next activities relative to MainMenu
    mainmenu_activity["Next Activity"]    = g["Activity Name"].shift(-1)
    mainmenu_activity["Second Activity"]  = g["Activity Name"].shift(-2)
    mainmenu_activity["Third Activity"]   = g["Activity Name"].shift(-3)

    # Keep only rows where current is MainMenu
    out = (
        mainmenu_activity.loc[mainmenu_activity["Activity Name"].eq("MainMenu"),
            ["Contact Session ID", "Date", "hour", "Next Activity", "Second Activity", "Third Activity", "DayOfWeek"]]
        .reset_index(drop=True)
    )

    # Rename "Next Activity" → "Current Activity"
    out = out.rename(columns={"Next Activity": "Current Activity"})

    # Clean up activity labels
    cols = ["Current Activity", "Second Activity", "Third Activity"]
    for c in cols:
        out[c] = (
            out[c].astype("string")
            .str.replace("SeniorsMenu", "Pre-Legal", regex=False)
            .str.replace("SuburbanPre-Legal", "SuburbanSeniors", regex=False)
            .str.replace("LegalMenu1", "Non-Senior Legal", regex=False)
            .str.replace("GetLoggedInSubSeniorConsumerAgents", "Senior Consumer Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorBenefitsAgents", "Senior Benefits Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorFamilySPAgents", "Senior Family Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorOtherSPAgents", "Senior Other Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorConsumerSPAgents", "Senior Consumer Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorADAPTAgents", "Senior ADAPT Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorHomeownerAgents", "Senior Homeowner Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Senior Employment Queue", regex=False)
            .str.replace("GetLoggedInConsumerSubSeniorsAgents", "Senior Consumer Queue", regex=False)
            .str.replace("GetLoggedInADAPTSPAgents',", "ADAPT Queue", regex=False)
            .str.replace("SeniorsADAPTMenu", "Senior ADAPT", regex=False)
            .str.replace("GetLoggedInEmploymentSubSeniorsAgents", "Senior Employment Queue", regex=False)
            .str.replace("GetLoggedInOtherSubSeniorsAgents", "Senior Other Queue", regex=False)
            .str.replace("GetLoggedInADAPTSubSeniorsAgents", "Senior ADAPT Queue", regex=False)
            .str.replace("GetLoggedInADAPTSubSeniorsSPAgents", "Senior ADAPT Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorADAPTAgents", "Senior ADAPT Queue", regex=False)
            .str.replace("GetLoggedInEmploymentSubSeniorsSPAgents", "Senior Employment Queue", regex=False)
            .str.replace("GetLoggedInHomeownerSubSeniorsAgents", "Senior Homeowner Queue", regex=False)
            .str.replace("GetLoggedInFamilySubSeniorsAgents", "Senior Family Queue", regex=False)
            .str.replace("GetLoggedInFamilySPAgents", "Senior Family Queue", regex=False)
            .str.replace("GetLoggedInBenefitsSubSeniorsSPAgents", "Senior Benefits Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorOtherAgents", "Senior Other Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorConsumerAgents", "Senior Consumer Queue", regex=False)
            .str.replace("GetLoggedInBenefitsSubSeniorsAgents", "Senior Benefits Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorConsumerAgents", "Senior Consumer Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorHomeownerSPAgents", "Senior Homeowner Queue", regex=False)
            .str.replace("GetLoggedInADAPTAgents", "ADAPT Queue", regex=False)
            .str.replace("GetLoggedInADAPTSPAgents", "ADAPT Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Senior Employment Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorADAPTSPAgents", "Senior ADAPT Queue", regex=False)
            .str.replace("BenefitsSP", "Consumer", regex=False)
            .str.replace("EmploymentSP", "Employment", regex=False)
            .str.replace("ImmigrationSP", "Immigration", regex=False)
            .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
            .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
            .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
            .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
            .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
            .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
            .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorFamilyAgents", "SubSenior Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Senior Employment Queue", regex=False)
            .str.replace("GetLoggedInConsumerSPAgents", "Consumer Queue", regex=False)
            .str.replace("GetLoggedInSubSeniorConsumerAgents", "Senior Consumer Queue", regex=False)
            .str.replace("SeniorNotCookCoMenu", "Non-Cook Seniors", regex=False)
            .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
            .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
            .str.replace("BenefitsMenu", "Benefits", regex=False)
            .str.replace("FamilyMenu", "Family", regex=False)
            .str.replace("HIVMenu", "HIV", regex=False)
            .str.replace("HousingMenu", "Housing", regex=False)
            .str.replace("ImmigrationMenu", "Immigration", regex=False)
            .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
            .str.replace("EmploymentMenu", "Employment", regex=False)
            .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
            .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
            .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
            .str.replace("ChildSupportMenu", "Child Support", regex=False)
            .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
            .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
            .str.replace("FrontDeskTransfer2", "FrontDeskTransfer", regex=False)
            .str.replace("FrontDeskTransfer3", "FrontDeskTransfer", regex=False)
        )

    return out


In [47]:
out_main = dash_data('Main Number Telephony EP')

out_main.loc[out_main['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_main.loc[out_main['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_main.to_csv(adhoc_data_path + 'main_dash.csv', index = False)